# CPI Prediction — Preprocessing & EDA

Loads the four raw CSVs from `data/raw/`, merges them into a monthly panel,
checks data quality, and produces exploratory plots.

**Outputs:**
- `data/processed/monthly_panel.csv` — input for `analysis.R`
- `output/figures/fig_timeseries.png` — CPI vs. predictors over time
- `output/figures/fig_correlations.png` — predictor correlations with next-month CPI

**Run `01_data_collection.ipynb` first.**

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
%matplotlib inline

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

os.makedirs('data/processed', exist_ok=True)
os.makedirs('output/figures', exist_ok=True)
os.makedirs('output/tables',  exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Working directory:', os.getcwd())

## 1. Load & Merge Raw Data

In [ ]:
def load_raw(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index) + pd.offsets.MonthEnd(0)
    df.index.name = 'date'
    return df

fred   = load_raw('data/raw/fred_data.csv')
spf    = load_raw('data/raw/spf_data.csv')
kalshi = load_raw('data/raw/kalshi_cpi.csv')
sent   = load_raw('data/raw/sentiment_data.csv')

panel = fred.join([spf, kalshi, sent], how='left')

# Drop raw price levels (only needed for YoY computation, already done)
panel = panel.drop(columns=['cpi_level', 'pce_level'], errors='ignore')

# Outcome variable: next-month CPI YoY (shift backward 1 period — no look-ahead bias)
panel['cpi_yoy_next'] = panel['cpi_yoy'].shift(-1)

# Restrict to analysis window: Jan 2022 – Mar 2026 (last month with a valid t+1 outcome)
panel = panel.loc['2022-01-31':'2026-03-31']
panel = panel.dropna(subset=['cpi_yoy_next'])

panel.to_csv('data/processed/monthly_panel.csv')
print(f'Panel saved: {panel.shape[0]} rows x {panel.shape[1]} cols')
print(f'Date range:  {panel.index[0].date()} to {panel.index[-1].date()}')
panel.head(3)

## 2. Data Quality

In [ ]:
print('Missing values per column:')
missing = panel.isnull().sum().rename('NaN count')
missing_pct = (panel.isnull().mean() * 100).round(1).rename('% missing')
display(pd.concat([missing, missing_pct], axis=1))

print('\nDescriptive statistics:')
display(panel.describe().round(3))

## 3. Time Series: Actual CPI vs. Key Predictors

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(panel.index, panel['cpi_yoy'], color='black', linewidth=2.2,
        label='Actual CPI YoY', zorder=5)
ax.plot(panel.index, panel['michigan_inflation'], color='#2196F3', linewidth=1.4,
        linestyle='--', label='Michigan Survey', alpha=0.85)
ax.plot(panel.index, panel['spf_cpi_forecast'], color='#1565C0', linewidth=1.4,
        linestyle=':', label='SPF Forecast', alpha=0.85)

if panel['kalshi_implied_cpi'].notna().sum() > 3:
    ax.plot(panel.index, panel['kalshi_implied_cpi'], color='#FF5722', linewidth=1.6,
            label='Kalshi Implied CPI', alpha=0.9)
else:
    ax.annotate('Kalshi: add credentials\nto 01_data_collection.ipynb',
                xy=(0.72, 0.88), xycoords='axes fraction',
                color='#FF5722', fontsize=9, style='italic')

ax.fill_between(panel.index,
                panel['cpi_yoy'].min() - 0.5, panel['cpi_yoy'].max() + 0.5,
                where=(panel['cpi_yoy'] > 5), alpha=0.06, color='red', label='CPI > 5%')

ax.set_ylabel('Percent (%)')
ax.set_title('CPI YoY vs. Prediction Sources (Jan 2022 – Mar 2026)', fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator(interval=4))
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig('output/figures/fig_timeseries.png', bbox_inches='tight')
plt.show()
print('Saved output/figures/fig_timeseries.png')

## 4. Correlations with Next-Month CPI

In [ ]:
MACRO_VARS     = ['michigan_inflation', 'cleveland_inflation', 'spf_cpi_forecast',
                  'unemployment', 'ppi', 'oil_wti', 'pce_yoy']
MARKET_VARS    = ['kalshi_implied_cpi']
SENTIMENT_VARS = ['gdelt_tone', 'google_trends_inflation']
ALL_VARS       = MACRO_VARS + MARKET_VARS + SENTIMENT_VARS

CATEGORY_MAP = {v: 'Macro/Survey'      for v in MACRO_VARS}
CATEGORY_MAP.update({v: 'Prediction Market' for v in MARKET_VARS})
CATEGORY_MAP.update({v: 'Sentiment'         for v in SENTIMENT_VARS})
CAT_COLORS = {'Macro/Survey': '#2196F3', 'Prediction Market': '#FF5722', 'Sentiment': '#4CAF50'}

cor_data = panel[['cpi_yoy_next'] + ALL_VARS].dropna()
cor_vals = cor_data.corr()['cpi_yoy_next'].drop('cpi_yoy_next')

cor_df = pd.DataFrame({
    'Variable':    cor_vals.index,
    'Category':    [CATEGORY_MAP[v] for v in cor_vals.index],
    'Correlation': cor_vals.values.round(3)
}).sort_values('Correlation', key=abs, ascending=False).reset_index(drop=True)

cor_df.to_csv('output/tables/correlation_table.csv', index=False)
display(cor_df)

fig, ax = plt.subplots(figsize=(9, 5))
colors = [CAT_COLORS[c] for c in cor_df['Category']]
ax.barh(cor_df['Variable'], cor_df['Correlation'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with CPI YoY (t+1)')
ax.set_title('Predictor Correlations with Next-Month CPI', fontweight='bold')
legend_handles = [mpatches.Patch(facecolor=v, label=k) for k, v in CAT_COLORS.items()]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('output/figures/fig_correlations.png', bbox_inches='tight')
plt.show()
print('Saved output/figures/fig_correlations.png')

## Done

| Output | Location |
|--------|----------|
| Merged monthly panel | `data/processed/monthly_panel.csv` |
| Correlation table | `output/tables/correlation_table.csv` |
| Time series plot | `output/figures/fig_timeseries.png` |
| Correlations plot | `output/figures/fig_correlations.png` |

**Next step:** Run `Rscript analysis.R` (or open it in RStudio) to run OLS and LASSO and generate all final tables and figures.